In [ ]:
# CELL 1 - mount + installs  (~1-2 min)
from google.colab import drive
drive.mount('/content/drive')
import sys, os
os.system(f"{sys.executable} -m pip -q install hnswlib sentence-transformers")
print("ready")


In [ ]:
# ============================================================
#  IS RETRIEVAL SILENTLY DEGRADED BY AN UNSET ef ?
# ============================================================
import os, json, pickle, hnswlib, numpy as np
from sentence_transformers import SentenceTransformer

ROOT = "/content/drive/MyDrive"
IDX  = f"{ROOT}/Phase1_Project/index_rebuild_base_verses_only/index"

model   = SentenceTransformer("Omartificial-Intelligence-Space/GATE-AraBert-v1")
entries = pickle.load(open(f"{IDX}/entries.pkl", "rb"))
ix = hnswlib.Index(space="cosine", dim=768)
ix.load_index(f"{IDX}/verses.hnsw")
print(f"index: {len(entries):,} entries")
print(f"hnswlib default ef on a freshly loaded index: {ix.ef}")

def search(q_emb, k, ef):
    ix.set_ef(max(ef, k))
    lab, dist = ix.knn_query(q_emb, k=k)
    return [entries[i]["verse_key"] for i in lab[0]], (1.0 - dist[0]).tolist()

def enc(qs):
    return model.encode(qs, convert_to_numpy=True, normalize_embeddings=True,
                        batch_size=32, show_progress_bar=False)

# ---------------------------------------------------------------- PART 1
print("\n" + "#" * 74)
print("# PART 1 - REPRODUCE THE ONE QUERY THAT DID NOT MATCH")
print("#" * 74)
Q3 = "من هو النبي المعروف بالصبر"
SAVED = ["69:40", "81:19", "33:45", "26:125", "26:143"]
SAVED_TOPSIM = 0.5121996998786926
e = enc([Q3])
print(f"\nquery: {Q3}")
print(f"saved run gave: {SAVED}  top-sim {SAVED_TOPSIM:.10f}\n")
for ef in [10, 16, 32, 64, 128, 256, 512]:
    keys, sims = search(e, 5, ef)
    m = "<== REPRODUCES THE SAVED RUN" if keys == SAVED and abs(sims[0]-SAVED_TOPSIM) < 1e-5 else ""
    print(f"  ef={ef:<4d} top5 {keys}")
    print(f"           top-sim {sims[0]:.10f}  {m}")

# ---------------------------------------------------------------- PART 2
print("\n" + "#" * 74)
print("# PART 2 - HOW OFTEN DOES ef=10 MISS THE TRUE NEAREST VERSE?")
print("#" * 74)
AY = None
for c in [f"{ROOT}/Phase4_Project/data/ayatec_records.json",
          f"{ROOT}/Phase2_Project/Roma_output/quranNLP/shared/data/ayatec_records.json"]:
    if os.path.exists(c):
        AY = c; break
print("ayatec:", AY)

recs = json.load(open(AY, encoding="utf-8"))
allv = {e_["verse_key"] for e_ in entries}
qs = [r for r in recs if r.get("verse_keys") and any(v in allv for v in r["verse_keys"])]
print(f"{len(recs)} records -> {len(qs)} usable questions with gold verses in this index")

embs = enc([r["question"] for r in qs])
print("encoded.")

# exact brute-force ground truth
mat = np.vstack([e_ for e_ in embs])
allemb = None
print("computing exact nearest neighbours by brute force for comparison ...")
# rebuild exact vectors from the index
vecs = np.array(ix.get_items(list(range(len(entries)))), dtype=np.float32)
vecs /= (np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-12)
exact_scores = mat @ vecs.T
exact_top1 = exact_scores.argmax(axis=1)
print("done.")

for ef in [10, 32, 64, 128, 256]:
    ix.set_ef(max(ef, 10))
    lab, _ = ix.knn_query(mat, k=10)
    miss1 = sum(1 for i in range(len(qs)) if lab[i][0] != exact_top1[i])
    print(f"  ef={ef:<4d}  top-1 differs from exact on {miss1:>3d}/{len(qs)} questions "
          f"({miss1/len(qs):.1%})")

# ---------------------------------------------------------------- PART 3
print("\n" + "#" * 74)
print("# PART 3 - WHAT IT COSTS ON THE AyaTEC BENCHMARK (k=10)")
print("#" * 74)
gold = [set(r["verse_keys"]) & allv for r in qs]

def evaluate(ef, k=10):
    ix.set_ef(max(ef, k))
    lab, _ = ix.knn_query(mat, k=k)
    rec = hit = mrr = 0.0
    for i, g in enumerate(gold):
        got = [entries[j]["verse_key"] for j in lab[i]]
        inter = set(got) & g
        rec += len(inter) / len(g)
        hit += 1.0 if inter else 0.0
        for r, vk in enumerate(got, 1):
            if vk in g:
                mrr += 1.0 / r
                break
    n = len(gold)
    return rec/n, hit/n, mrr/n

print(f"\n  {'ef':<6} {'Recall@10':>10} {'HitRate@10':>11} {'MRR@10':>9}")
rows = {}
for ef in [10, 32, 64, 128, 256, 512]:
    r, h, m = evaluate(ef)
    rows[ef] = (r, h, m)
    print(f"  {ef:<6} {r:>10.4f} {h:>11.4f} {m:>9.4f}")

r10, h10, m10 = rows[10]
r512, h512, m512 = rows[512]
print(f"\n  going from the default ef=10 to ef=512:")
for name, a, b in [("Recall@10", r10, r512), ("HitRate@10", h10, h512), ("MRR@10", m10, m512)]:
    d = (b - a) / a * 100 if a else float("nan")
    print(f"    {name:<11} {a:.4f} -> {b:.4f}   ({d:+.1f}%)")
print("\n  (this is free - it is a search-time parameter, no retraining, no reindexing)")

print("\n" + "=" * 74)
print("DONE")
print("=" * 74)
